Trabajo Práctico Grupal · Cámara de Diputados, 2022–2026

Grid de 10 gráficos (5 años × 2 métodos) a partir de `idps_py/idps.csv`
(generado por [`03_puntos_ideales.py`](03_puntos_ideales.py), que calcula los
puntos ideales con dos métodos: **PCA** (el atajo, primera componente
principal) y **Bayes** (el modelo de Clinton, Jackman y Rivers, implementado
en [`ideal_pscl.py`](ideal_pscl.py)).

Cada panel es un gráfico de barras divergentes: una fila por legislador,
ordenado de izquierda (azul, `idp < 0`) a derecha (rojo, `idp > 0`). En los
paneles de Bayes, una banda gris detrás de la barra muestra `idp_bayes ±
sd_bayes` — la incertidumbre de la estimación. Pasar el mouse sobre cualquier
barra muestra el nombre del legislador y su valor.

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy.special import log_ndtr

import igraph as ig
import matplotlib.pyplot as plt
from adjustText import adjust_text
from matplotlib.lines import Line2D

## Cargar los puntos ideales

In [ ]:
df = pd.read_csv("datos/idps.csv")
df.head()

## Orientar el PCA

El signo de `idp_pca` es arbitrario por año (lo decide sklearn en
`03_puntos_ideales.py`, no los datos — ver `pca_atajo()`). Lo reorientamos acá
contra `idp_bayes`, que sí viene fijado contra legisladores de referencia de
izquierda/derecha (`orientar()` en `03_puntos_ideales.py`): si la correlación
entre los dos métodos da negativa para un año, se invierte el signo de
`idp_pca` ese año.

In [ ]:
def orientar_pca(df):
    df = df.copy()
    for anio, g in df.groupby("anio"):
        r = g["idp_pca"].corr(g["idp_bayes"])
        if r is not None and r < 0:
            df.loc[g.index, "idp_pca"] = -df.loc[g.index, "idp_pca"]
    return df


df = orientar_pca(df)

## Armar el grid de 10 paneles

In [ ]:
IZQUIERDA, DERECHA = "#2a78d6", "#e34948"
WHISKER, BASELINE, GRID = "#898781", "#c3c2b7", "#e1e0d9"
SURFACE, PAGE = "#fcfcfb", "#f9f9f7"
TEXT, TEXT2, MUTED = "#0b0b0b", "#52514e", "#898781"
FONT = "system-ui, -apple-system, \'Segoe UI\', sans-serif"


def panel(fig, row, col, nombres, valores, sd=None):
    """Un panel: barras divergentes desde 0, ordenadas por valor, con
    opcionalmente un whisker +-sd (Bayes) y hover de nombre + idp."""
    orden = valores.argsort()
    v, nom = valores.to_numpy()[orden], nombres.to_numpy()[orden]
    colores = [DERECHA if x >= 0 else IZQUIERDA for x in v]
    kw = dict(
        x=v, y=list(range(len(v))), orientation="h",
        marker=dict(color=colores, line=dict(width=0)),
        showlegend=False,
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    )
    if sd is not None:
        s = sd.to_numpy()[orden]
        kw["error_x"] = dict(type="data", array=s, color=WHISKER, thickness=1, width=0)
        kw["customdata"] = list(zip(nom, s))
        kw["hovertemplate"] = "<b>%{customdata[0]}</b><br>idp = %{x:.2f} (\u00b1%{customdata[1]:.2f})<extra></extra>"
    else:
        kw["customdata"] = nom
        kw["hovertemplate"] = "<b>%{customdata}</b><br>idp = %{x:.2f}<extra></extra>"
    fig.add_trace(go.Bar(**kw), row=row, col=col)

In [ ]:
anios = sorted(df["anio"].unique())

bayes_domain = math.ceil((df["idp_bayes"].abs() + df["sd_bayes"]).max() * 10) / 10
pca_domain = 1.08

fig = make_subplots(
    rows=len(anios), cols=2,
    column_titles=["Bayes · media posterior", "PCA · reescalado a [-1, 1], orientado contra Bayes"],
    row_titles=[str(a) for a in anios],
    horizontal_spacing=0.05, vertical_spacing=0.03,
)

for i, anio in enumerate(anios, start=1):
    g = df[df["anio"] == anio]
    panel(fig, i, 1, g["nombre"], g["idp_bayes"], sd=g["sd_bayes"])
    gp = g.dropna(subset=["idp_pca"])
    panel(fig, i, 2, gp["nombre"], gp["idp_pca"])

# leyenda compartida: trazas invisibles, una sola vez
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                          marker=dict(size=10, color=IZQUIERDA),
                          name="Izquierda (idp < 0)"), row=1, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                          marker=dict(size=10, color=DERECHA),
                          name="Derecha (idp > 0)"), row=1, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                          line=dict(color=WHISKER, width=2),
                          name="\u00b1 1 desvio (incertidumbre, solo Bayes)"), row=1, col=1)

for i in range(1, len(anios) + 1):
    fig.update_xaxes(range=[-bayes_domain, bayes_domain], zeroline=True,
                      zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                      tickfont=dict(size=10, color=MUTED), row=i, col=1)
    fig.update_xaxes(range=[-pca_domain, pca_domain], zeroline=True,
                      zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                      tickfont=dict(size=10, color=MUTED), row=i, col=2)
    for c in (1, 2):
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False,
                          autorange="reversed", row=i, col=c)

fig.update_annotations(font=dict(size=13, color=TEXT2))
for ann in fig.layout.annotations:
    if ann.text in [str(a) for a in anios]:
        ann.font = dict(size=15, color=TEXT)

fig.update_layout(
    title=dict(text="Puntos ideales \u00b7 C\u00e1mara de Diputados, 2022\u20132026", x=0.01,
               font=dict(size=20, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=1500, width=1000,
    margin=dict(l=70, r=20, t=110, b=30),
    legend=dict(x=1.0, y=1.0, xanchor="left", yanchor="top", font=dict(size=11)),
    bargap=0.1,
)

fig.show(config={"displaylogo": False, "scrollZoom": False})

PENDIENTE: Anotar cosas interesantes acá. Como el desvío estandar de Massa en el 2022 (equisdé)

PENDIENTE: IDEA - Inspirado en lo anterior, una tabla con los diputados con más desvío por año.. y cómo interpretamos eso en el mundo real

## Dot plot (Wilkinson): distribución por bloque

Otra forma de ver lo mismo: en vez de ordenar a cada legislador en su propia
fila (como arriba), se los apila — cada legislador es un círculo, y los que
caen en el mismo entorno de `idp_bayes` se amontonan uno sobre otro. El perfil
que arma la pila se lee como un histograma, pero sin perder la unidad
individual (sigue siendo "un legislador, un punto"). Es el **dot plot** de
Wilkinson (*"Dot Plots"*, 1999) — también se lo conoce como *dot histogram*.
No es lo mismo que un Cleveland dot plot (una marca por categoría, eje
categórico) ni que un beeswarm (jitter continuo, sin bins): acá el eje x está
discretizado en bins de ancho fijo y el apilamiento es vertical y exacto.

**Una aclaración sobre el color:** esto está coloreado por `bloque` real (el
bloque de cada legislador ese año, tomado de
`legisladores_bloque_provincia.csv`), no por una reconstrucción de los clusters/etiquetas que
se ven en la imagen de referencia — esa segmentación (p. ej. que UxP aparezca
partido en varios grupos separados) no se puede reconstruir de forma confiable
a partir de una captura de pantalla, así que en vez de inventar un criterio de
clustering fui con el dato real y verificable que tenemos: a qué bloque
pertenece cada uno.

In [ ]:
ANIO_DOT = 2026
ANCHO_BIN = 0.06

bloques = pd.read_csv("datos/legisladores_bloque_provincia.csv")
bloque_legislador = bloques[bloques["yr"] == ANIO_DOT].set_index("legislador")["bloque"]

dd = df[df["anio"] == ANIO_DOT].copy()
dd["bloque"] = dd["nombre"].map(bloque_legislador)
dd = dd.dropna(subset=["bloque"]).reset_index(drop=True)
dd["bloque"].value_counts()

Los tres bloques más chicos de izquierda (PTS, Partido Obrero e Izquierda
Socialista, 1 o 2 bancas cada uno) se agrupan bajo "Frente de Izquierda" —así
se los suele nombrar en conjunto— para que el color tenga sentido como
categoría. El resto de los bloques chicos (menos de 10 legisladores cada uno)
va a "Otros bloques". Evito reusar azul/rojo acá: en los gráficos de arriba
esos dos colores ya significan izquierda/derecha (polaridad), y esto es una
identidad categórica distinta (bloque), así que mezclar los dos códigos de
color confundiría más de lo que aclara.

In [ ]:
IZQUIERDA_MICRO = [
    "Pts-frente De Izquierda Y De Trabajadores Unidad",
    "Partido Obrero En El Frente De Izquierda Y De Trabajadores-unidad",
    "Izquierda Socialista - Frente De Izquierda Unidad",
]
dd["bloque"] = dd["bloque"].replace({b: "Frente de Izquierda" for b in IZQUIERDA_MICRO})

PRINCIPALES = ["Frente de Izquierda", "Union Por La Patria", "Provincias Unidas",
               "Innovacion Federal", "Pro", "La Libertad Avanza"]
dd["bloque_mostrado"] = dd["bloque"].where(dd["bloque"].isin(PRINCIPALES), "Otros bloques")

COLOR_BLOQUE = {
    "Frente de Izquierda": "#4a3aa7",
    "Union Por La Patria": "#eb6834",
    "Provincias Unidas": "#1baf7a",
    "Innovacion Federal": "#eda100",
    "Pro": "#e87ba4",
    "La Libertad Avanza": "#008300",
    "Otros bloques": "#c3c2b7",
}

In [ ]:
def apilar(valores, ancho_bin):
    """Wilkinson: redondea cada valor al bin mas cercano y devuelve, para cada
    punto, en que nivel de la pila de ese bin le toca (0, 1, 2, ...)."""
    binned = (valores / ancho_bin).round() * ancho_bin
    nivel = binned.groupby(binned).cumcount()
    return binned, nivel


dd = dd.sort_values("idp_bayes").reset_index(drop=True)
dd["x_bin"], dd["nivel"] = apilar(dd["idp_bayes"], ANCHO_BIN)

gaps = dd["idp_bayes"].diff()
i_max = gaps.idxmax()
hueco = gaps.loc[i_max]
x_hueco = (dd.loc[i_max - 1, "idp_bayes"] + dd.loc[i_max, "idp_bayes"]) / 2
print(f"mayor hueco: {hueco:.2f}, entre {dd.loc[i_max - 1, 'nombre']} y {dd.loc[i_max, 'nombre']}")

In [ ]:
fig2 = go.Figure()

for bloque_val, g in dd.groupby("bloque_mostrado"):
    fig2.add_trace(go.Scatter(
        x=g["x_bin"], y=g["nivel"],
        mode="markers",
        marker=dict(size=9, color=COLOR_BLOQUE[bloque_val], line=dict(width=0)),
        name=f"{bloque_val} ({len(g)})",
        customdata=list(zip(g["nombre"], g["bloque"], g["idp_bayes"])),
        hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>idp = %{customdata[2]:.2f}<extra></extra>",
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    ))

fig2.add_vline(x=x_hueco, line=dict(color=MUTED, width=1, dash="dot"))
fig2.add_annotation(x=x_hueco, y=dd["nivel"].max() * 0.85,
                     text=f"mayor hueco: {hueco:.2f}", showarrow=False,
                     font=dict(size=11, color=TEXT2), xanchor="left", xshift=6)

fig2.update_xaxes(zeroline=True, zerolinecolor=BASELINE, zerolinewidth=1,
                   gridcolor=GRID, tickfont=dict(size=11, color=MUTED))
fig2.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)

fig2.update_layout(
    title=dict(text=f"Puntos ideales (Bayes) por bloque \u00b7 C\u00e1mara de Diputados, {ANIO_DOT}",
               x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=480, width=1000,
    margin=dict(l=40, r=20, t=70, b=40),
    legend=dict(orientation="h", y=-0.15, x=0),
)

fig2.show(config={"displaylogo": False, "scrollZoom": False})

PENDIENTE - Hacer el mismo grafico (Dot Plot de Wikinson) para el resto de los años 
PENDIENTE - El hueco más grande en 0.71, qué significa eso en el mundo real? Que entre Grabois y Giordano se encuentra la brecha ideológica más grande y , por lo tanto, el camino más lejano para unir?

PENDIENTE - IDEA: Así como tenemos el espectro de diputados... Podríamos hacer un espectro de proyectos. Desde el 2022 al 2026, hacer un dotplot similar al de arriba pero ubicando los proyectos de acuerdo a que tan "de derecha" son o que tan "de izquierda" son. Esto se saca por la adhesión que tuvieron en la derecha/izquierda y rechazo que tuvieron en derecha/izquierda.

PENDIENTE - IDEA: Proyectos que no respetaron la polaridad derecha-izquierda. Es decir, proyectos que fueron muy divisivos pero tiene a una cantidad relevante de gente de izquierda votando igual que derecha. En clase nombraron la ley de aborto, por ejemplo, como un proyecto que tiene ejes o dimensiones que escapan al espectro derecha izquierda. A mi se me ocurre ley de presupuesto general (puede haber gente de izquierda a favor para evitar la discresionalidad del poder Ejecutivo VS gente de izquierda en contra porque avala el ajuste) 

## Mapa de proyectos: corte, discriminación y dirección

El modelo de Clinton, Jackman y Rivers no solo ubica legisladores: cada votación
*j* también tiene sus parámetros, en `P(Sí) = Φ(β_j · θ_i − α_j)`:

- **Corte** `α_j / β_j`: el punto del eje izquierda–derecha donde un legislador
  queda indiferente entre Sí y No. Cerca de 0 es una votación que parte la Cámara
  al medio; lejos de 0 es una votación donde solo un extremo quedó del otro lado
  (qué tan *extrema* es).
- **Discriminación** `|β_j|`: qué tan nítido es ese corte. Alta = el voto se
  explica casi del todo por la ideología (izquierda de un lado, derecha del
  otro); cerca de 0 = el eje izquierda–derecha no explica el voto (otros clivajes:
  territoriales, de conciencia, etc.).
- **Dirección** (color): el `idp_bayes` promedio de los que votaron **Sí**. Rojo =
  la apoyó la derecha; azul = la apoyó la izquierda. Es otra forma de leer el
  signo de β, en unidades de punto ideal.

`ideal_pscl.py` muestrea β y α pero no los guarda, así que acá se reestiman sin
volver a correr la cadena: para cada votación, un probit de los votos (Sí = 1,
No = 0, el resto afuera) contra los `idp_bayes` ya estimados, con el mismo prior
N(0, 25) del modelo (estimación MAP). El prior es también el que pone techo a β
en las votaciones con separación perfecta (≈ 11). Las votaciones unánimes quedan
afuera, igual que en `preparar_pscl()`.

In [ ]:
ses = pd.read_csv("datos/sesiones_dt.csv.gz", dtype={"legislador": str}, keep_default_na=False)
info = pd.read_csv("datos/info_sesiones_dt.csv")

votos = (ses.assign(y=ses["voto"].map({"AFIRMATIVO": 1.0, "NEGATIVO": 0.0}))
            .dropna(subset=["y"])
            .merge(df[["nombre", "anio", "idp_bayes"]], left_on=["legislador", "yr"], right_on=["nombre", "anio"]))


def parametros_votacion(g):
    x, y = g["idp_bayes"].to_numpy(), g["y"].to_numpy()
    if y.min() == y.max():
        return None
    s = 2 * y - 1
    menos_log_post = lambda p: -log_ndtr(s * (p[0] * x - p[1])).sum() + 0.02 * (p ** 2).sum()
    beta, alpha = minimize(menos_log_post, [0.0, 0.0], method="BFGS").x
    return pd.Series({"beta": beta, "alpha": alpha, "corte": alpha / beta,
                      "discriminacion": abs(beta), "idp_si": x[y == 1].mean(),
                      "afirmativos": int(y.sum()), "negativos": int(len(y) - y.sum())})


proyectos = (votos.groupby(["anio", "id_votacion"]).apply(parametros_votacion).dropna().reset_index()
                  .merge(info[["id_votacion", "titulo", "fecha", "resultado"]], on="id_votacion"))
proyectos.describe().round(2)

In [ ]:
RANGO_CORTE = 2.5

visibles = proyectos[proyectos["corte"].abs() <= RANGO_CORTE]
print(f"{len(proyectos) - len(visibles)} de {len(proyectos)} votaciones con |corte| > {RANGO_CORTE} quedan fuera "
      f"del gráfico (casi unánimes, con discriminación baja: el corte no está bien definido)")

fig3 = go.Figure()
for anio in anios:
    g = visibles[visibles["anio"] == anio]
    fig3.add_trace(go.Scatter(
        x=g["corte"], y=g["discriminacion"], mode="markers", name=str(anio),
        marker=dict(size=9, color=g["idp_si"], colorscale=[[0, IZQUIERDA], [0.5, GRID], [1, DERECHA]],
                    cmin=-1, cmax=1, cmid=0, line=dict(width=1, color=SURFACE),
                    colorbar=dict(title=dict(text="idp medio<br>de los Sí", font=dict(size=11, color=TEXT2)),
                                  tickvals=[-1, 0, 1], ticktext=["izquierda", "centro", "derecha"],
                                  tickfont=dict(size=10, color=MUTED), thickness=12, len=0.6)),
        customdata=list(zip(g["titulo"], g["fecha"], g["resultado"], g["idp_si"], g["afirmativos"], g["negativos"])),
        hovertemplate=("<b>%{customdata[0]}</b><br>%{customdata[1]} · %{customdata[2]} "
                       "(%{customdata[4]:.0f} Sí / %{customdata[5]:.0f} No)<br>"
                       "corte = %{x:.2f} · discriminación = %{y:.2f} · idp medio Sí = %{customdata[3]:.2f}<extra></extra>"),
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    ))

botones = [dict(label="Todos", method="update", args=[{"visible": [True] * len(anios)}])]
botones += [dict(label=str(a), method="update", args=[{"visible": [b == a for b in anios]}]) for a in anios]

fig3.update_xaxes(title=dict(text="corte (α/β) · ← izquierda · derecha →", font=dict(size=12, color=TEXT2)),
                  range=[-RANGO_CORTE, RANGO_CORTE], zeroline=True, zerolinecolor=BASELINE,
                  gridcolor=GRID, tickfont=dict(size=11, color=MUTED))
fig3.update_yaxes(title=dict(text="discriminación |β|", font=dict(size=12, color=TEXT2)),
                  rangemode="tozero", gridcolor=GRID, zeroline=False, tickfont=dict(size=11, color=MUTED))
fig3.update_layout(
    title=dict(text="Votaciones 2022–2026: dónde cortan, qué tan nítido y quién las apoyó",
               x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=600, width=1000, showlegend=False,
    margin=dict(l=60, r=20, t=110, b=60),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, x=0.01, xanchor="left",
                      y=1.02, yanchor="bottom", showactive=True, pad=dict(r=4, t=0),
                      bgcolor=SURFACE, bordercolor=GRID, font=dict(size=11, color=TEXT2))],
)
fig3.show(config={"displaylogo": False, "scrollZoom": False})

Cómo leerlo: arriba = votaciones puramente ideológicas (el voto se predice
casi perfecto con el punto ideal); abajo = votaciones que el eje
izquierda–derecha no explica — ahí hay que buscar los proyectos que "no
respetaron la polaridad" de la idea de arriba (aborto, presupuesto, etc.). El
corte en el eje x dice dónde parte la Cámara: cerca de 0, mitad y mitad; hacia
los costados, solo un extremo quedó del otro lado.

## Grafos de cercanía y comunidades

Lo mismo que `04_redes.py` (el script de redes de la materia), pero dibujado acá
en vez de en HTML. Se parte de `datos/distancias_idps.csv.gz`: el producto
cartesiano de diputados de cada año, con la distancia entre sus puntos ideales
(`|idp_bayes A - idp_bayes B|`).

Para cada año:

1. Se arma la red completa: un nodo por diputado y una arista por par, con la
   distancia como peso. Sobre esa red se calcula la **intermediación**
   (betweenness) de cada diputado.
2. Se **poda**: se borran las aristas con distancia mayor a `PUNTO_CORTE` (0.4),
   y después los diputados que quedan con menos de 3 uniones.
3. Se detectan **comunidades** con fast greedy (peso `1 / distancia`, así una
   unión corta pesa más), cortando donde la modularidad es máxima.

En el dibujo, el color es la comunidad (las 8 más grandes; el resto en gris), el
tamaño del nodo es su intermediación y se etiquetan los 5 diputados con más
intermediación. La leyenda dice, para cada comunidad, qué bloque es mayoría y en
qué proporción. La posición sale de Fruchterman-Reingold, así que la forma
cambia un poco entre corridas (la estructura no).

In [ ]:
PUNTO_CORTE = 0.4
COLORES_COMUNIDAD = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
OTRAS, ARISTA = "#c3c2b7", "#898781"

distancias = pd.read_csv("datos/distancias_idps.csv.gz").rename(
    columns={"nombre1": "Source", "nombre2": "Target", "distancia": "weight"})
distancias = distancias[distancias.Source < distancias.Target]


def armar_grafo(anio):
    dist = distancias[distancias.anio == anio]
    dips = bloques[bloques.yr == anio]
    g = ig.Graph.DataFrame(dist[["Source", "Target", "weight"]], directed=False,
                           vertices=dips[["legislador"]], use_vids=False)
    g.vs["bloque"] = list(dips.bloque)
    return g


def podar(g):
    g = g.copy()
    g.simplify(combine_edges="sum")
    g.delete_edges(g.es.select(weight_gt=PUNTO_CORTE))
    g.simplify(combine_edges="sum")
    g.delete_vertices(g.vs.select(_degree_lt=3))
    return g


def intermediacion(g):
    n = g.vcount()
    return [b * 2 / ((n - 1) * (n - 2)) for b in g.betweenness(directed=False, weights="weight")]


def comunidades(g):
    arbol = g.community_fastgreedy(weights=[1 / w for w in g.es["weight"]])
    membresia = ig.community_to_membership(arbol.merges, g.vcount(), g.vcount() - arbol.optimal_count)
    return ig.VertexClustering(g, membresia)

In [ ]:
def dibujar_red(anio):
    g = armar_grafo(anio)
    g.vs["betweenness"] = intermediacion(g)
    g = podar(g)
    x = comunidades(g)
    por_tamanio = sorted(range(len(x)), key=lambda c: -len(x[c]))
    color = {c: COLORES_COMUNIDAD[k] if k < len(COLORES_COMUNIDAD) else OTRAS for k, c in enumerate(por_tamanio)}

    fig, ax = plt.subplots(figsize=(13, 8), facecolor=SURFACE)
    xs, ys = zip(*g.layout_fruchterman_reingold(weights=[1 / w for w in g.es["weight"]], niter=2000).coords)
    for e in g.es:
        a, b = e.tuple
        ax.plot([xs[a], xs[b]], [ys[a], ys[b]], color=ARISTA, alpha=0.05, lw=0.5, zorder=1)
    ax.scatter(xs, ys, s=[10 + 500 * b for b in g.vs["betweenness"]], c=[color[c] for c in x.membership],
               edgecolors=SURFACE, linewidths=0.6, zorder=2)

    top = sorted(range(g.vcount()), key=lambda i: -g.vs[i]["betweenness"])[:5]
    textos = [ax.text(xs[i], ys[i], g.vs[i]["name"].replace('"', "").title(), fontsize=8, color=TEXT, zorder=3)
              for i in top]
    adjust_text(textos, ax=ax, expand=(2, 2.5), arrowprops=dict(arrowstyle="-", color=ARISTA, lw=0.5))

    leyenda = []
    for c in por_tamanio[:len(COLORES_COMUNIDAD)]:
        miembros = pd.Series(g.vs[x[c]]["bloque"])
        bloque = miembros.mode()[0]
        leyenda.append(Line2D([], [], marker="o", ls="", markersize=8, markerfacecolor=color[c], markeredgewidth=0,
                              label=f"{len(miembros)} dip. · {(miembros == bloque).mean():.0%} {bloque}"))
    if len(x) > len(COLORES_COMUNIDAD):
        resto = sum(len(x[c]) for c in por_tamanio[len(COLORES_COMUNIDAD):])
        leyenda.append(Line2D([], [], marker="o", ls="", markersize=8, markerfacecolor=OTRAS, markeredgewidth=0,
                              label=f"{resto} dip. · otras {len(x) - len(COLORES_COMUNIDAD)} comunidades"))
    ax.legend(handles=leyenda, loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False, fontsize=9,
              labelcolor=TEXT2, title="Comunidad · bloque mayoritario", title_fontsize=9, alignment="left")
    ax.set_title(f"{anio} · {g.vcount()} diputados, {g.ecount()} uniones, {len(x)} comunidades",
                 loc="left", fontsize=14, color=TEXT)
    ax.set_axis_off()
    plt.show()

PENDIENTE - IDEA: Les gusta la idea de una nube de palabras?? Teniendo los títulos de los proyectos, podemos hacer una nube de los proyectos con mayor adhesión derecha, y una nube con proyectos con mayor adhesion izquierda

In [ ]:
dibujar_red(2022)

In [ ]:
dibujar_red(2023)

In [ ]:
dibujar_red(2024)

En 2025 hay más diputados que bancas (257) porque el 10 de diciembre cambió la
composición de la Cámara: en `legisladores_bloque_provincia.csv` aparecen tanto
los que salieron como los que entraron ese año.

In [ ]:
dibujar_red(2025)

In [ ]:
dibujar_red(2026)

PENDIENTE : Los graficos de redes quedaron horribles. Habrá que hacerlos en HTML o sino trabajar directamente con comunidades